
## Project: New Zealand Health Access & Equity Reporting Dashboard

This notebook performs the initial data profiling for the Ministry of Health New Zealand Health Survey datasets.

The purpose of this step is to understand the structure, quality and reporting potential of the raw datasets before cleaning, modelling and loading the data into PostgreSQL.

The project focuses only on "health access barriers", including access issues related to cost, appointment availability, prescription collection, after-hours care, dental care and other service access indicators where available.

## Data Sources

The raw datasets were downloaded from the Ministry of Health New Zealand Health Survey data explorer.https://minhealthnz.shinyapps.io/nz-health-survey-2024-25-annual-data-explorer/_w_8b722c5d613943158c146c963f8da469/#!/download-data-sets

Three CSV files are used:

1. `prevalence_mean.csv`  
   Contains prevalence and mean estimates for the total population and subgroups.

2. `subgroup_comparison.csv`  
   Contains adjusted prevalence or mean ratios comparing selected population groups.

3. `changes_over_time.csv`  
   Contains prevalence and mean estimates across survey years, including time-series results where available.

The original files are stored in `data/raw/` and should not be edited directly.

# 01 Data Profiling

In [2]:
import pandas as pd
from pathlib import Path

raw_path = Path("../data/raw")

prevalence = pd.read_csv(raw_path / "prevalence_mean.csv")
comparison = pd.read_csv(raw_path / "subgroup_comparison.csv")
trend = pd.read_csv(raw_path / "changes_over_time.csv")

print("Files loaded successfully.")

Files loaded successfully.


## Check dataset size and columns

This step checks the number of rows, columns and field names in each raw file.

In [3]:
datasets = {
    "prevalence_mean": prevalence,
    "subgroup_comparison": comparison,
    "changes_over_time": trend
}

for name, df in datasets.items():
    print(f"\n{name}")
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")
    print("Column names:")
    print(list(df.columns))


prevalence_mean
Rows: 47708
Columns: 19
Column names:
['population', 'short.description', 'year', 'group', 'total', 'flag_for_publishing', 'total.low.CI', 'total.high.CI', 'male', 'male_flag_for_publishing', 'male.low.CI', 'male.high.CI', 'female', 'female_flag_for_publishing', 'female.low.CI', 'female.high.CI', 'estimated.number', 'estimated.number.low.CI', 'estimated.number.high.CI']

subgroup_comparison
Rows: 3198
Columns: 8
Column names:
['population', 'short.description', 'year', 'comparison', 'adjusted.rate.ratio', 'adjusted.rate.ratio.low.CI', 'adjusted.rate.ratio.high.CI', 'adjusted.for']

changes_over_time
Rows: 9041
Columns: 31
Column names:
['Unnamed: 0', 'population', 'group', 'short.description', 'percent.11', 'percent.12', 'percent.13', 'percent.14', 'percent.15', 'percent.16', 'percent.17', 'percent.18', 'percent.19', 'percent.20', 'percent.21', 'percent.22', 'percent.23', 'percent.24', 'p.value.24.11', 'p.value.24.12', 'p.value.24.13', 'p.value.24.14', 'p.value.24.15',

## Review sample records and missing values

This step shows sample rows and checks missing values.

In [4]:
prevalence.head()

,population,short.description,year,group,total,flag_for_publishing,total.low.CI,total.high.CI,male,male_flag_for_publishing,male.low.CI,male.high.CI,female,female_flag_for_publishing,female.low.CI,female.high.CI,estimated.number,estimated.number.low.CI,estimated.number.high.CI
0,adults,"Good, very good, or excellent self-rated health",2011,Total,89.3,NaN,88.5,90.0,89.6,NaN,88.4,90.7,89.0,NaN,88.2,89.7,3117000.0,3091000.0,3142000.0
1,adults,"Good, very good, or excellent self-rated health",2012,Total,89.5,NaN,88.8,90.2,89.3,NaN,88.2,90.3,89.7,NaN,88.9,90.5,3151000.0,3129000.0,3174000.0
2,adults,"Good, very good, or excellent self-rated health",2013,Total,91.4,NaN,90.7,92.0,91.3,NaN,90.4,92.1,91.4,NaN,90.6,92.2,3269000.0,3246000.0,3291000.0
3,adults,"Good, very good, or excellent self-rated health",2014,Total,88.9,NaN,88.1,89.7,89.1,NaN,88.0,90.2,88.7,NaN,87.6,89.7,3255000.0,3226000.0,3284000.0
4,adults,"Good, very good, or excellent self-rated health",2015,Total,87.7,NaN,87.1,88.3,87.9,NaN,87.0,88.8,87.5,NaN,86.7,88.3,3294000.0,3271000.0,3318000.0


In [5]:
comparison.head()

,population,short.description,year,comparison,adjusted.rate.ratio,adjusted.rate.ratio.low.CI,adjusted.rate.ratio.high.CI,adjusted.for
0,children,"Good, very good, or excellent parent-rated health",2024,Boys vs girls,1.00,0.99,1.02,Age
1,children,"Good, very good, or excellent parent-rated health",2024,Māori vs non-Māori,1.00,0.99,1.02,"Age, gender"
2,children,"Good, very good, or excellent parent-rated health",2024,Māori boys vs non-Māori boys,1.00,0.98,1.02,Age
3,children,"Good, very good, or excellent parent-rated health",2024,Māori girls vs non-Māori girls,1.01,0.98,1.03,Age
4,children,"Good, very good, or excellent parent-rated health",2024,Pacific vs non-Pacific,0.99,0.96,1.01,"Age, gender"


In [6]:
trend.head()

,Unnamed: 0,population,group,short.description,percent.11,percent.12,percent.13,percent.14,percent.15,percent.16,...,p.value.24.14,p.value.24.15,p.value.24.16,p.value.24.17,p.value.24.18,p.value.24.19,p.value.24.20,p.value.24.21,p.value.24.22,p.value.24.23
0,1,children,Total,"Good, very good, or excellent parent-rated health",97.9,98.2,98.4,97.9,97.6,98.0,...,0.3995,0.9491,0.3288,0.2362,0.1840,0.9334,0.9905,0.6185,0.0201,0.1095
1,2,children,Boys,"Good, very good, or excellent parent-rated health",97.5,98.0,97.9,97.8,97.7,97.8,...,0.8810,0.9554,0.7663,0.5952,0.7848,0.7209,0.4160,0.5519,0.0965,0.2427
2,3,children,Girls,"Good, very good, or excellent parent-rated health",98.3,98.4,98.9,98.1,97.4,98.1,...,0.2580,0.9781,0.2741,0.2365,0.1052,0.6238,0.3013,0.1143,0.1336,0.2649
3,4,children,0-4,"Good, very good, or excellent parent-rated health",97.5,98.2,98.2,97.9,98.0,98.3,...,0.9711,0.8705,0.5225,0.8956,0.2938,0.6992,0.9996,0.9640,0.0807,0.0959
4,5,children,'5-9,"Good, very good, or excellent parent-rated health",98.2,98.8,98.7,98.2,97.2,97.6,...,0.3841,0.9582,0.7672,0.5122,0.0594,0.6682,0.8623,0.0317,0.7186,0.5996


## Identify Health Access Indicators

The `short.description` column contains the health indicator name.

This step searches for indicators related to healthcare access barriers, such as GP access, prescription cost, dental care, appointment availability, after-hours care and unmet need.

Only relevant health access indicators will be selected for this project.

In [8]:
# Check how many unique indicators are in each dataset

for name, df in datasets.items():
    unique_count = df["short.description"].nunique()
    print(f"{name}: {unique_count} unique indicators")

prevalence_mean: 195 unique indicators
subgroup_comparison: 195 unique indicators
changes_over_time: 195 unique indicators


In [9]:
# Search for possible health access indicators

keywords = [
    "GP",
    "general practitioner",
    "medical centre",
    "appointment",
    "prescription",
    "dental",
    "after-hours",
    "after hours",
    "unmet need",
    "cost",
    "unable"
]

def search_indicators(df, dataset_name):
    results = df[
        df["short.description"]
        .astype(str)
        .str.contains("|".join(keywords), case=False, na=False)
    ]["short.description"].drop_duplicates().sort_values()
    
    print(f"\n{dataset_name}: {len(results)} possible indicators found")
    for indicator in results:
        print("-", indicator)

search_indicators(prevalence, "prevalence_mean")
search_indicators(comparison, "subgroup_comparison")
search_indicators(trend, "changes_over_time")


prevalence_mean: 24 possible indicators found
- Consulted GP or nurse about mental health
- Consulted a nurse or GP about mental health
- Dental health care worker visit
- Dietitian at usual medical centre
- GP at usual medical centre
- GP visit
- Has usual medical centre
- Last GP visit (any location) free
- Mean number of GP visits
- Mental health worker at usual medical centre
- Nurse at usual medical centre
- Only visit dental health care worker for problems
- Physiotherapist at usual medical centre
- Unfilled prescription due to cost
- Unmet need for GP due to care for a dependent
- Unmet need for GP due to cost
- Unmet need for GP due to dislike
- Unmet need for GP due to lack of supporter
- Unmet need for GP due to owing money
- Unmet need for GP due to transport
- Unmet need for GP due to wait time
- Unmet need for GP due to work
- Unmet need for dental health care due to cost
- Unmet need for mental health care and addictions services

subgroup_comparison: 24 possible indicat

## Review Candidate Health Access Indicators

The keyword search found possible health access indicators.  
To review them more clearly, the results are combined into one table with the dataset name and indicator name.

In [10]:
# Create a clearer table of possible health access indicators

keywords = [
    "GP",
    "general practitioner",
    "medical centre",
    "appointment",
    "prescription",
    "dental",
    "after-hours",
    "after hours",
    "unmet need",
    "cost",
    "unable"
]

candidate_rows = []

for dataset_name, df in datasets.items():
    matched_indicators = (
        df[
            df["short.description"]
            .astype(str)
            .str.contains("|".join(keywords), case=False, na=False)
        ]["short.description"]
        .drop_duplicates()
        .sort_values()
    )
    
    for indicator in matched_indicators:
        candidate_rows.append({
            "dataset": dataset_name,
            "indicator": indicator
        })

candidate_indicators = pd.DataFrame(candidate_rows)

candidate_indicators

,dataset,indicator
0,prevalence_mean,Consulted GP or nurse about mental health
1,prevalence_mean,Consulted a nurse or GP about mental health
2,prevalence_mean,Dental health care worker visit
3,prevalence_mean,Dietitian at usual medical centre
4,prevalence_mean,GP at usual medical centre
...,...,...
67,changes_over_time,Unmet need for GP due to transport
68,changes_over_time,Unmet need for GP due to wait time
69,changes_over_time,Unmet need for GP due to work
70,changes_over_time,Unmet need for dental health care due to cost


In [11]:
# Show unique candidate indicators across all three datasets

unique_candidate_indicators = (
    candidate_indicators["indicator"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

unique_candidate_indicators.to_frame(name="candidate_indicator")

,candidate_indicator
0,Consulted GP or nurse about mental health
1,Consulted a nurse or GP about mental health
2,Dental health care worker visit
3,Dietitian at usual medical centre
4,GP at usual medical centre
5,GP visit
6,Has usual medical centre
7,Last GP visit (any location) free
8,Mean number of GP visits
9,Mental health worker at usual medical centre


In [12]:
# Show full indicator names without truncation

pd.set_option("display.max_colwidth", None)

unique_candidate_indicators.to_frame(name="candidate_indicator")

,candidate_indicator
0,Consulted GP or nurse about mental health
1,Consulted a nurse or GP about mental health
2,Dental health care worker visit
3,Dietitian at usual medical centre
4,GP at usual medical centre
5,GP visit
6,Has usual medical centre
7,Last GP visit (any location) free
8,Mean number of GP visits
9,Mental health worker at usual medical centre


## Check Selected Indicators Across Datasets

This step checks whether the selected health access barrier indicators are available in all three datasets.

This is important because the dashboard needs:

- prevalence data to show how common each barrier is
- subgroup comparison data to analyse equity gaps
- trend data to show whether barriers are improving or worsening over time

In [13]:
selected_indicators = [
    "Unfilled prescription due to cost",
    "Unmet need for GP due to cost",
    "Unmet need for GP due to transport",
    "Unmet need for GP due to wait time",
    "Unmet need for GP due to work",
    "Unmet need for dental health care due to cost",
    "Unmet need for mental health care and addictions services"
]

selected_indicators

['Unfilled prescription due to cost',
 'Unmet need for GP due to cost',
 'Unmet need for GP due to transport',
 'Unmet need for GP due to wait time',
 'Unmet need for GP due to work',
 'Unmet need for dental health care due to cost',
 'Unmet need for mental health care and addictions services']

In [14]:
# Check whether selected indicators exist in each dataset

for dataset_name, df in datasets.items():
    available_indicators = set(df["short.description"].unique())
    
    print(f"\n{dataset_name}")
    for indicator in selected_indicators:
        if indicator in available_indicators:
            print(f"Available - {indicator}")
        else:
            print(f"Missing   - {indicator}")


prevalence_mean
Available - Unfilled prescription due to cost
Available - Unmet need for GP due to cost
Available - Unmet need for GP due to transport
Available - Unmet need for GP due to wait time
Available - Unmet need for GP due to work
Available - Unmet need for dental health care due to cost
Available - Unmet need for mental health care and addictions services

subgroup_comparison
Available - Unfilled prescription due to cost
Available - Unmet need for GP due to cost
Available - Unmet need for GP due to transport
Available - Unmet need for GP due to wait time
Available - Unmet need for GP due to work
Available - Unmet need for dental health care due to cost
Available - Unmet need for mental health care and addictions services

changes_over_time
Available - Unfilled prescription due to cost
Available - Unmet need for GP due to cost
Available - Unmet need for GP due to transport
Available - Unmet need for GP due to wait time
Available - Unmet need for GP due to work
Available - Unm

## Create Indicator Mapping

The indicator mapping table defines the final project scope.

It keeps the selected health access barrier indicators and assigns each one a short name and barrier type. This makes the project easier to understand in PostgreSQL, Power BI and the final dashboard.

In [15]:
indicator_mapping = pd.DataFrame([
    {
        "indicator_id": 1,
        "indicator_name": "Unfilled prescription due to cost",
        "indicator_short_name": "Prescription cost barrier",
        "theme": "Health Access",
        "barrier_type": "Cost",
        "include_flag": True
    },
    {
        "indicator_id": 2,
        "indicator_name": "Unmet need for GP due to cost",
        "indicator_short_name": "GP cost barrier",
        "theme": "Health Access",
        "barrier_type": "Cost",
        "include_flag": True
    },
    {
        "indicator_id": 3,
        "indicator_name": "Unmet need for GP due to transport",
        "indicator_short_name": "GP transport barrier",
        "theme": "Health Access",
        "barrier_type": "Transport",
        "include_flag": True
    },
    {
        "indicator_id": 4,
        "indicator_name": "Unmet need for GP due to wait time",
        "indicator_short_name": "GP wait time barrier",
        "theme": "Health Access",
        "barrier_type": "Timeliness",
        "include_flag": True
    },
    {
        "indicator_id": 5,
        "indicator_name": "Unmet need for GP due to work",
        "indicator_short_name": "GP work constraint",
        "theme": "Health Access",
        "barrier_type": "Time constraint",
        "include_flag": True
    },
    {
        "indicator_id": 6,
        "indicator_name": "Unmet need for dental health care due to cost",
        "indicator_short_name": "Dental cost barrier",
        "theme": "Health Access",
        "barrier_type": "Cost",
        "include_flag": True
    },
    {
        "indicator_id": 7,
        "indicator_name": "Unmet need for mental health care and addictions services",
        "indicator_short_name": "Mental health access barrier",
        "theme": "Health Access",
        "barrier_type": "Mental health access",
        "include_flag": True
    }
])

indicator_mapping

,indicator_id,indicator_name,indicator_short_name,theme,barrier_type,include_flag
0,1,Unfilled prescription due to cost,Prescription cost barrier,Health Access,Cost,True
1,2,Unmet need for GP due to cost,GP cost barrier,Health Access,Cost,True
2,3,Unmet need for GP due to transport,GP transport barrier,Health Access,Transport,True
3,4,Unmet need for GP due to wait time,GP wait time barrier,Health Access,Timeliness,True
4,5,Unmet need for GP due to work,GP work constraint,Health Access,Time constraint,True
5,6,Unmet need for dental health care due to cost,Dental cost barrier,Health Access,Cost,True
6,7,Unmet need for mental health care and addictions services,Mental health access barrier,Health Access,Mental health access,True


In [16]:
# Save indicator mapping table

indicator_mapping.to_csv("../config/indicator_mapping.csv", index=False)

print("indicator_mapping.csv saved successfully.")

indicator_mapping.csv saved successfully.
